In [0]:
from pyspark.sql.functions import col ,lit , current_timestamp, max , min , sum , count , trim , lower ,upper 

In [0]:
bronze_table = "globalretail.bronze.customers_raw"
silver_table ="globalretail.silver.dim_customers"
hwm_table = "globalretail.validation.silver_watermarks"


hwm_last_loaded_at = spark.sql(f"select last_successful_load_at from {hwm_table} where entity_name='customers'").collect()[0][0]

max_last_loaded_at = spark.sql(f"select max(ingested_at) from {bronze_table}").collect()[0][0]

print(hwm_last_loaded_at)

print(max_last_loaded_at)

bronze_df = spark.read.table(bronze_table)

new_row_df = bronze_df.filter(col("ingested_at") > hwm_last_loaded_at)

display(new_row_df)


In [0]:
new_row_df.createOrReplaceTempView("vw_customer_from_bronze")




In [0]:
%sql
select *  from vw_customer_from_bronze   ;

In [0]:
latest_after_transformation = spark.sql("""
                                        
                                        
        select * , LOWER(trim(email)) as email_clean,

        CASE 
           WHEN upper(trim(country_code))='USA' THEN 'US'
           WHEN upper(trim(country_code))='UK' THEN 'GB'
           WHEN upper(trim(country_code))='IND' THEN 'IN'
           ELSE upper(trim(country_code))
        
        END as country_code_std,

        CASE 
           WHEN upper(trim(loyalty_tier))='PLATNIUM' THEN 'PLatinum'
           WHEN upper(trim(loyalty_tier))='GOLD' THEN 'Gold'
           WHEN upper(trim(loyalty_tier))='SILVER' THEN 'Silver'
           ELSE 'Bronze'

        END AS loyalty_tier_std,


        CASE 
           WHEN upper(trim(marketing_opt_in)) IN ('Y','YES','T','TRUE') THEN TRUE
           ELSE false
        END AS marketing_opt_in_std,

        CONCAT_WS(' ',TRIM(first_name),TRIM(last_name)) as full_name,

        CASE
           WHEN REGEXP_EXTRACT(
            LOWER(TRIM(email)),
            '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$',
            0
            ) != '' THEN 'VALID'
           ELSE 'INVALID_EMAIL'
        END AS dq_status

        from vw_customer_from_bronze 
        """)

In [0]:

rejects = latest_after_transformation.filter(col("dq_status") != "VALID").select(
    col("customer_id").cast("bigint"),
    col("email_clean").alias("email"),
    col("country_code_std").alias("country_code"),
    col("dq_status").alias("reject_reason"),
    current_timestamp().alias("rejected_at"),
)
reject_count = rejects.count()
if reject_count > 0:
    rejects.write.mode("append").saveAsTable("globalretail.silver.customers_rejects")

In [0]:
valid_latest = latest_after_transformation.filter(col("dq_status") == "VALID").select(
    col("customer_id").cast("bigint"),
    "full_name",
    col("email_clean").alias("email"),
    "phone",
    "address_line1",
    "city",
    "state",
    col("country_code_std").alias("country_code"),
    "postal_code",
    col("loyalty_tier_std").alias("loyalty_tier"),
    col("marketing_opt_in_std").alias("marketing_opt_in"),
    "source_system",
    col("updated_at").cast("timestamp").alias("src_updated_at"),
)

In [0]:
display(valid_latest)

In [0]:
valid_latest.createOrReplaceTempView("vw_customer_source")

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW vw_customer_changes AS 

SELECT s.* , 
CASE
   WHEN  t.customer_id IS NULL THEN 'NEW'
   WHEN s.city <> t.city 
          OR s.state <> t.state
          OR s.loyalty_tier <> t.loyalty_tier 
          OR s.source_system <> t.source_system 
        THEN 'CHANGED'
   ELSE 'NO_CHANGE'
END AS chnage_type
from vw_customer_source as s  
left join globalretail.silver.dim_customers as t
on s.customer_id = t.customer_id and t.is_current =true;
            

In [0]:
%sql

MERGE INTO globalretail.silver.dim_customers t
USING vw_customer_changes c
ON t.customer_id = c.customer_id AND t.is_current= True 
AND c.chnage_type = 'CHANGED'

WHEN MATCHED THEN UPDATE SET 
t.effective_end_ts = c.src_updated_at - INTERVAL 1 SECOND,
t.is_current = False,
t.updated_at = current_timestamp()



In [0]:
%sql 

INSERT INTO globalretail.silver.dim_customers 
SELECT 
abs(xxhash64(concat(cast(customer_id AS STRING), '||', cast(src_updated_at AS STRING)))) AS customer_sk,
customer_id,
full_name,
email,
phone,
address_line1,
city,
state,
country_code,
postal_code,
loyalty_tier,
marketing_opt_in,
source_system,
src_updated_at AS updated_at,
TIMESTAMP('9999-12-31 23:59:59') as effective_end_ts,
True AS is_current,
'Valid' AS dq_status,
current_timestamp() AS created_at,
current_timestamp() AS updated_at
FROM vw_customer_changes
WHERE chnage_type in ('NEW','CHANGED')


In [0]:
spark.sql(f"""
UPDATE globalretail.validation.silver_watermarks
SET  last_successful_load_at = TIMESTAMP('{max_last_loaded_at}')
WHERE entity_name = 'customers'
""")

In [0]:
%sql
select * from globalretail.silver.dim_customers
where customer_id=1034;